# 03. Chunking Strategies:
Chunking is the physical process of breaking continuous documents down into retrievable vectors. The core dilemma of chunking is balancing two conflicting forces:

**Precision:** You want small chunks (e.g., 100 to 250 tokens) so the vector embedding captures a single, highly specific concept without dilution.

**Context:** You want large chunks (e.g., 1,000 to 2,000 tokens) so the LLM has enough surrounding background narrative to avoid hallucinations.

## 1. Breakdown of Production Chunking Strategies
**Recursive Character Chunking (The Robust Baseline):**
It splits text using a hierarchical list of separators (typically ["\n\n", "\n", " ", ""]). It tries to break text at natural paragraph boundaries first, falling back to sentences or words only if a block exceeds the target size limit.

**Semantic Chunking (The Topic-Aware Splitter):**
Instead of arbitrary lengths, it computes the embedding distance between consecutive sentences. When the semantic distance crosses a specific threshold, it cuts a chunk boundary, ensuring each chunk represents a cohesive single thought or topic.

**Parent-Child (Hierarchical) Chunking (The Production Standard):**
It solves the precision-vs-context conflict directly. It divides documents into Small Child Chunks (optimized for tight vector search matching) that link back via metadata keys to Large Parent Chunks (fed to the LLM for rich generation context).

**Late Chunking:**
A modern technique where you pass the entire long document through a long-context transformer model first to preserve token-level contextual representations, and then pool vectors per chunk so that every chunk carries global document awareness.

## 2. Implementation Code: Parent-Child Hierarchical Chunker
Create this script in your repository under: 07-RAG/03_chunking_strategies/parent_child_chunker.py

In [ ]:
"""
Module 03: Parent-Child Chunking Strategy
Demonstrates hierarchical text splitting to maximize vector search precision 
while retaining broad parent contexts for LLM generation.
"""

from typing import List, Dict, Any
import uuid

class ParentChildChunker:
    def __init__(self, parent_size: int = 1000, child_size: int = 250):
        self.parent_size = parent_size
        self.child_size = child_size

    def split_document(self, document_text: str, source_metadata: Dict[str, Any]) -> Dict[str, Any]:
        """
        Simulates splitting a source document into large parent nodes 
        and breaking those parents down into granular child nodes.
        """
        # Step 1: Create coarse parent chunks (simulated paragraph/section splits)
        parent_chunks = self._chunk_text(document_text, self.parent_size)
        
        hierarchical_store = {
            "parents": {},  # Key: parent_id, Value: Full text context
            "children": []  # List of child objects mapped to parent_id
        }

        for parent_text in parent_chunks:
            parent_id = str(uuid.uuid4())
            
            # Store parent text in mock key-value doc store
            hierarchical_store["parents"][parent_id] = {
                "text": parent_text,
                "metadata": source_metadata
            }

            # Step 2: Subdivide parent into precise child chunks for vector indexing
            child_texts = self._chunk_text(parent_text, self.child_size)
            for child_text in child_texts:
                child_id = str(uuid.uuid4())
                hierarchical_store["children"].append({
                    "child_id": child_id,
                    "parent_id": parent_id,
                    "child_text": child_text,
                    "metadata": source_metadata
                })

        return hierarchical_store

    def _chunk_text(self, text: str, max_chars: int) -> List[str]:
        """Recursive helper simulation to segment text by max character limits."""
        words = text.split()
        chunks = []
        current_chunk = []
        current_length = 0

        for word in words:
            current_length += len(word) + 1
            if current_length > max_chars:
                chunks.append(" ".join(current_chunk))
                current_chunk = [word]
                current_length = len(word)
            else:
                current_chunk.append(word)
        
        if current_chunk:
            chunks.append(" ".join(current_chunk))
            
        return chunks


# Example execution block
if __name__ == "__main__":
    sample_enterprise_text = (
        "Generative AI infrastructure requires meticulous data engineering. "
        "When implementing Retrieval-Augmented Generation, developers often face performance bottlenecks "
        "due to poor document parsing. By adopting hierarchical storage designs like parent-child chunking, "
        "systems can achieve high-precision vector searches without sacrificing the narrative background "
        "required by downstream large language models."
    )

    chunker = ParentChildChunker(parent_size=200, child_size=80)
    result = chunker.split_document(
        document_text=sample_enterprise_text, 
        source_metadata={"source": "architecture_guide.pdf", "author": "Platform Team"}
    )

    print(f"Total Parents Created: {len(result['parents'])}")
    print(f"Total Children Created for Vector DB: {len(result['children'])}")
    print("\nSample Child Node Mapping:")
    print(result['children'][0])

### Senior Developer Interview Spotlight
Q: Why might standard fixed-size chunking degrade RAG performance on code files or complex legal contracts?

Answer: Fixed-size chunking is blind to syntax. In code files, it can slice a function right in half or split a class definition, cutting off imports or variable definitions. In legal documents, it can sever conditional clauses from their governing definitions (e.g., separating "Party A shall pay..." from "...unless Clause 4 applies"). This causes semantic misalignment during embedding generation, leading directly to retrieval failures and hallucinations.

Q: What are the trade-offs of using Semantic Chunking over Recursive Character Chunking?

Answer: Semantic chunking delivers superior chunk cohesion because boundaries align with actual topic shifts rather than arbitrary character lengths. However, its major trade-off is compute overhead and latency at index time: semantic chunking requires embedding every single sentence and computing matrix similarity distances across the entire document corpus before ingestion can even begin.

## Chunking startegies in detail

## 1. Fixed-Size Chunking
Fixed-size chunking splits text into chunks of a predetermined number of characters or tokens, often with an optional overlap.

How it works: You define a chunk size (e.g., 512 tokens) and a chunk overlap (e.g., 50 tokens). The overlap ensures that sentences spanning across chunk boundaries aren't abruptly cut off.

**Pros:**

Simple and computationally inexpensive.

Predictable chunk sizes (great for strict LLM token limits).

**Cons:**

Ignores semantic boundaries (can split a sentence or a paragraph right in the middle).

May separate a pronoun from its antecedent noun, reducing retrieval accuracy.

Python Example (using LangChain):

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = "Your long document text goes here..."
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)
chunks = text_splitter.split_text(text)

## 2. Recursive Chunking
Recursive chunking attempts to split text hierarchically using a list of separators (usually ["\n\n", "\n", " ", ""]) until the chunks are small enough.

How it works: It starts by trying to split paragraphs (\n\n). If a paragraph is still too large, it moves to the next separator (single newline \n), then spaces, and finally characters, until every chunk fits the target size.

**Pros:**

Keeps related paragraphs, sentences, and words together as much as possible.

The industry standard default for most text-heavy RAG pipelines.

**Cons:**

Slightly more complex than fixed-size chunking.

Does not look at the meaning of the text, only structural separators.

Python Example (using LangChain):

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.create_documents([text])

## 3. Semantic Chunking
Semantic chunking determines chunk boundaries dynamically by measuring the statistical distance between consecutive sentences.

**How it works:**

The document is split into individual sentences.

Embeddings are generated for each sentence.

The cosine distance is calculated between consecutive sentences.

A threshold (e.g., percentile-based, standard deviation) is set. If the distance between two sentences exceeds the threshold, a new chunk boundary is created.

**Pros:**

Ensures that semantic topics stay together within the same chunk.

Adapts naturally to the flow of information in the document.

**Cons:**

Computationally expensive (requires generating embeddings for every single sentence upfront).

Slower processing time during ingestion.

Python Example (using LangChain):

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# Uses embedding distance to create semantic boundaries
text_splitter = SemanticChunker(OpenAIEmbeddings())
chunks = text_splitter.create_documents([text])

## 4. Parent-Child (Hierarchical) Chunking
Parent-child chunking addresses a fundamental RAG dilemma: small chunks are great for precise vector retrieval, but large chunks are better for giving the LLM enough context to generate an answer.

**How it works:**

Documents are split into large Parent Chunks (e.g., 2000 tokens).

Each Parent Chunk is further subdivided into smaller Child Chunks (e.g., 400 tokens).

Only the Child Chunks are embedded and stored in the vector database.

During retrieval, the system searches and matches the user query against the Child Chunks, but instead of passing the child to the LLM, it fetches and passes the corresponding Parent Chunk.

**Pros:**

High retrieval precision (thanks to small child chunks).

Rich context for generation (thanks to large parent chunks).

**Cons:**

Requires more storage and a more intricate retrieval architecture.

**Architecture Flow:**

#Quick comparision table 

| Strategy | Best Used For | Pros | Cons
| :--- | :--- | :--- | :--- |
| Fixed-Size | "Simple text strict token budgets" | "Fast, predictable" | Breaks sentences/meaning
| Recursive | General purpose documents (Default) | Keeps structure intact |"Structure-dependent, not meaning-driven"
| Semantic | "Articles ,  essays context-heavy books" | Topic integrity | High compute cost
| Parent-Child | Complex documents requiring deep context | Best balance of precision & context | Complex setup